# Deliverable 3 — Dataset exploration and EDA

Audits the paired SAR images and masks, summarizes image dimensions and pixel statistics, and saves an inventory and sample grid **in this submission folder**. Set `DATA_ROOT` below if the dataset is stored somewhere other than the project's `data/raw` directory.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

def find_folder(name):
    candidates = [Path.cwd(), Path.cwd().parent / name, Path.cwd().parent.parent / name]
    for candidate in candidates:
        if candidate.name == name and candidate.is_dir():
            return candidate.resolve()
    raise FileNotFoundError(f"Could not locate sibling folder: {name}")

SUBMISSION_DIR = find_folder("oil_spill_detection_submission")
project_candidates = [Path.cwd(), Path.cwd().parent / "oil_spill_detection", Path.cwd().parent.parent / "oil_spill_detection"]
PROJECT_ROOT = next((p.resolve() for p in project_candidates if (p / "src" / "data.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the oil_spill_detection project folder. Open the notebook from that folder or set PROJECT_ROOT manually.")
sys.path.insert(0, str(PROJECT_ROOT))
from src.data import find_pairs, load_gray

DATA_ROOT = PROJECT_ROOT / "data" / "raw"  # Change this if your dataset is elsewhere.
OUTPUT_DIR = SUBMISSION_DIR / "deliverable_3_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Project:", PROJECT_ROOT)
print("Dataset:", DATA_ROOT)
print("EDA output:", OUTPUT_DIR)

In [ ]:
records = []
sample_pairs = []
for split in ("train", "val", "test"):
    try:
        pairs = find_pairs(DATA_ROOT, split)
    except FileNotFoundError:
        if split in ("train", "val"):
            raise
        print(f"Optional {split} split not found; skipping it.")
        continue
    print(f"{split}: {len(pairs)} image-mask pairs")
    sample_pairs.extend((split, image_path, mask_path) for image_path, mask_path in pairs[:4])
    for image_path, mask_path in pairs:
        image = load_gray(image_path)
        mask = load_gray(mask_path, mask=True)
        with Image.open(image_path) as source_image:
            width, height = source_image.size
        records.append({
            "split": split,
            "image": str(image_path),
            "mask": str(mask_path),
            "width": width,
            "height": height,
            "oil_pixel_fraction": float(mask.mean()),
            "image_mean": float(image.mean()),
            "image_std": float(image.std()),
        })

inventory = pd.DataFrame(records)
if inventory.empty:
    raise RuntimeError("No image-mask pairs were found. Check DATA_ROOT and the dataset layout.")
inventory.to_csv(OUTPUT_DIR / "image_inventory.csv", index=False)
display(inventory.groupby("split").agg(
    image_count=("image", "count"),
    mean_width=("width", "mean"),
    mean_height=("height", "mean"),
    mean_oil_pixel_fraction=("oil_pixel_fraction", "mean"),
))
display(inventory.head())
print("Inventory saved:", OUTPUT_DIR / "image_inventory.csv")

In [ ]:
n = min(len(sample_pairs), 8)
fig, axes = plt.subplots(n, 2, figsize=(8, max(3, 2.5 * n)), squeeze=False)
for row, (split, image_path, mask_path) in enumerate(sample_pairs[:n]):
    axes[row, 0].imshow(load_gray(image_path), cmap="gray")
    axes[row, 0].set_title(f"{split}: {image_path.name}")
    axes[row, 1].imshow(load_gray(mask_path, mask=True), cmap="gray")
    axes[row, 1].set_title("Ground-truth mask")
    axes[row, 0].axis("off")
    axes[row, 1].axis("off")
fig.tight_layout()
grid_path = OUTPUT_DIR / "sample_images_and_masks.png"
fig.savefig(grid_path, dpi=140, bbox_inches="tight")
plt.show()
print("Sample grid saved:", grid_path)